# Bonilla et al.: leakage-resistant CMAQ error rerun

This notebook is the auditable entry point for the replacement analysis. The complete implementation is in `leakage_resistant_pipeline.py` in the same folder; this notebook exposes the design, checks the saved outputs, and can rerun the workflow.

**Primary design:** predict daily CMAQ total PM₂.₅ error (`cmaq_total_pm25 - observed_pm25`) using site-year grouped splitting. The untouched test set contains 20% of site-year groups. Five-fold grouped cross-validation is performed only inside the training partition.

**Leakage controls:** `observed_pm25`, `cmaq_total_pm25`, `cmaq_nofire_pm25`, observed concentration regime, fire name, and fire-window tags are excluded from the primary feature matrix. Observed concentration regimes are used only after prediction for descriptive stratification.

**Extreme-value rule:** the 18 otherwise-valid rows with CMAQ total PM₂.₅ > 557 µg/m³ are quarantined from the primary analysis without capping or deletion from the source. They remain in the raw-data sensitivity analysis.

In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd()
if not (HERE / "leakage_resistant_pipeline.py").exists():
    HERE = Path("leakage_resistant_rerun")

CANONICAL_DIR = HERE
canonical_manifest = json.loads((CANONICAL_DIR / "run_manifest.json").read_text(encoding="utf-8"))
canonical_manifest

{'workflow': 'Bonilla et al leakage-resistant CMAQ error rerun',
 'random_seed': 20260905,
 'grouping_unit': 'site_id-year',
 'test_group_fraction': 0.2,
 'cv_folds': 5,
 'target': 'cmaq_total_pm25 - observed_pm25',
 'primary_features': ['cmaq_fire_pm25',
  'latitude',
  'longitude',
  'year',
  'month',
  'dayofyear',
  'distance_to_fire_km',
  'distance_to_fire_missing',
  'n_obs_records'],
 'prohibited_primary_features': ['observed_pm25',
  'cmaq_total_pm25',
  'cmaq_nofire_pm25',
  'pm25_regime',
  'fire_name',
  'fire_window_tag'],
 'primary_qc_rule': 'observed_pm25 in [0,557]; cmaq_total_pm25 <= 557; no capping/winsorization',
 'sensitivity_rule': 'retain all otherwise-valid rows, including cmaq_total_pm25 > 557',
 'n_full_hard_valid': 187069,
 'n_primary': 187051,
 'n_quarantined': 18,
 'n_groups': 2139,
 'n_test_groups': 428,
 'rf_parameters': {'n_estimators': 160,
  'max_depth': 24,
  'min_samples_leaf': 10,
  'max_features': 0.7,
  'n_jobs': -1,
  'random_state': 20260905},
 

## Reproduce the full analysis

`RUN_FULL_ANALYSIS` is set to `True` below. If the notebook has been moved, update `DATA_PATH`. The run is deterministic given the recorded seed and software versions. Regenerated files are written to a separate `reproduction_check` directory so the canonical results are not overwritten.

In [2]:
import sys

sys.path.insert(0, str(HERE.resolve()))
from leakage_resistant_pipeline import run

DATA_PATH = (HERE / "../data/paired_cmaq_aqs_fireseason_2008_2018.csv").resolve()
REPRODUCTION_DIR = HERE / "reproduction_check"
RUN_FULL_ANALYSIS = True

if RUN_FULL_ANALYSIS:
    run(DATA_PATH, REPRODUCTION_DIR)
    RESULTS_DIR = REPRODUCTION_DIR
else:
    RESULTS_DIR = CANONICAL_DIR
    print("Saved canonical results loaded.")

manifest = json.loads((RESULTS_DIR / "run_manifest.json").read_text(encoding="utf-8"))

{
  "status": "complete",
  "outdir": "C:\\Users\\boxem\\Documents\\Codex\\2026-09-04\\pls\\outputs\\leakage_resistant_rerun\\reproduction_check",
  "primary_rows": 187051,
  "test_rows": 37430,
  "quarantined": 18,
  "second_ale_feature": "longitude"
}


## Verify group isolation and QC accounting

These assertions are intentionally independent of the modelling code. They confirm that test groups do not occur in training, each training row has one CV fold, and all source rows passing hard validity are accounted for as primary or quarantined.

In [3]:
folds = pd.read_csv(RESULTS_DIR / "fold_assignments.csv", dtype={"site_id": str, "group_id": str})
train_groups = set(folds.loc[folds.split_role == "train", "group_id"])
test_groups = set(folds.loc[folds.split_role == "test", "group_id"])

assert train_groups.isdisjoint(test_groups)
assert (folds.loc[folds.split_role == "train", "cv_fold"].between(0, 4)).all()
assert len(folds) == manifest["n_full_hard_valid"]
assert (folds.qc_status == "quarantined_extreme_cmaq").sum() == manifest["n_quarantined"]

{
    "rows": len(folds),
    "groups": folds.group_id.nunique(),
    "training_groups": len(train_groups),
    "test_groups": len(test_groups),
    "quarantined_rows": int((folds.qc_status == "quarantined_extreme_cmaq").sum()),
}

{'rows': 187069,
 'groups': 2139,
 'training_groups': 1711,
 'test_groups': 428,
 'quarantined_rows': 18}

## Primary test results and baselines

In [4]:
comparison = pd.read_csv(RESULTS_DIR / "model_comparison_test_metrics.csv")
comparison

,analysis,model,feature_set,n,r2_coefficient_of_determination,pearson_r,rmse_ug_m3,mae_ug_m3,mean_residual_pred_minus_actual_ug_m3
0,primary,leakage_resistant_random_forest,cmaq_fire_pm25 | latitude | longitude | year |...,37430,0.710610,8.443776e-01,5.823815,3.435358,0.171117
1,primary,training_mean_baseline,none,37430,-0.000001,-1.977944e-17,10.825943,4.988096,-0.010926
2,primary,linear_regression,cmaq_fire_pm25 | latitude | longitude | year |...,37430,0.599740,7.782681e-01,6.849148,4.292821,0.133052
3,primary,fire_only_random_forest,cmaq_fire_pm25,37430,0.591765,7.695429e-01,6.917048,4.321572,0.101388
4,component_sensitivity,no_fire_component_sensitivity_rf,cmaq_nofire_pm25 | latitude | longitude | year...,37430,0.335434,5.806691e-01,8.825402,3.779793,0.034758
5,raw_extreme_sensitivity,random_forest_including_quarantined_extremes,cmaq_fire_pm25 | latitude | longitude | year |...,37433,0.381436,8.284539e-01,9.439691,3.508616,0.316278


## Five-fold grouped cross-validation

In [5]:
cv = pd.read_csv(RESULTS_DIR / "grouped_cv_metrics.csv")
display(cv)
cv[["r2_coefficient_of_determination", "rmse_ug_m3", "mae_ug_m3"]].agg(["mean", "std"])

,analysis,model,fold,n,r2_coefficient_of_determination,pearson_r,rmse_ug_m3,mae_ug_m3,mean_residual_pred_minus_actual_ug_m3,n_training_groups,n_validation_groups
0,primary_grouped_cv,random_forest,0,29926,0.546274,0.739885,6.316035,3.481521,0.274999,1368,343
1,primary_grouped_cv,random_forest,1,29924,0.628759,0.793113,5.752198,3.452035,0.065972,1369,342
2,primary_grouped_cv,random_forest,2,29923,0.647371,0.805507,5.939212,3.378269,0.146146,1369,342
3,primary_grouped_cv,random_forest,3,29922,0.453333,0.674749,7.225800,3.310109,0.244543,1369,342
4,primary_grouped_cv,random_forest,4,29926,0.628857,0.794104,5.717289,3.448327,0.300830,1369,342


,r2_coefficient_of_determination,rmse_ug_m3,mae_ug_m3
mean,0.580919,6.190107,3.414052
std,0.081365,0.625857,0.069380


## Untouched-test performance by observed regime (post hoc only)

In [6]:
pd.read_csv(RESULTS_DIR / "primary_test_metrics_by_regime.csv")

,analysis,regime,n,r2_coefficient_of_determination,pearson_r,rmse_ug_m3,mae_ug_m3,mean_residual_pred_minus_actual_ug_m3
0,primary_untouched_test,All,37430,0.710610,0.844378,5.823815,3.435358,0.171117
1,primary_untouched_test,Background (<12),25242,0.669708,0.837660,3.707031,2.498707,-1.146279
2,primary_untouched_test,Moderate (12-35),11715,0.729564,0.879852,6.652019,4.741773,2.331210
3,primary_untouched_test,High (>35),473,0.694975,0.897137,29.234345,21.063893,16.974988


## Independent-test permutation importance and ALE values

In [7]:
importance = pd.read_csv(RESULTS_DIR / "permutation_importance_test_set.csv")
ale = pd.read_csv(RESULTS_DIR / "ale_values.csv")
display(importance)
display(ale.groupby("feature").agg(n_bins=("bin", "size"), min_effect=("ale_effect_ug_m3", "min"), max_effect=("ale_effect_ug_m3", "max")))

,feature,importance_mean,importance_sd
0,cmaq_fire_pm25,1.101198e+00,1.473096e-02
1,longitude,1.072446e-01,9.018851e-03
2,latitude,9.332386e-02,6.418146e-04
3,year,6.844424e-02,2.831788e-03
4,dayofyear,6.312078e-02,2.883240e-03
5,month,6.324533e-03,6.045460e-04
6,distance_to_fire_km,6.056481e-03,5.771489e-04
7,distance_to_fire_missing,8.968130e-04,2.012737e-04
8,n_obs_records,5.551115e-17,5.551115e-17


,n_bins,min_effect,max_effect
feature,,,
cmaq_fire_pm25,20,-12.209173,219.797000
longitude,20,-2.653932,2.245963


## Reproduction comparison against the frozen canonical run

In [8]:
canonical_folds = pd.read_csv(CANONICAL_DIR / "fold_assignments.csv")
reproduced_folds = pd.read_csv(REPRODUCTION_DIR / "fold_assignments.csv")
pd.testing.assert_frame_equal(canonical_folds, reproduced_folds, check_dtype=False)

comparison_files = [
    "grouped_cv_metrics.csv",
    "grouped_cv_summary.csv",
    "model_comparison_test_metrics.csv",
    "primary_test_metrics_by_regime.csv",
    "permutation_importance_test_set.csv",
    "ale_values.csv",
    "Table_S3_regime_error_summary.csv",
]
for filename in comparison_files:
    expected = pd.read_csv(CANONICAL_DIR / filename)
    reproduced = pd.read_csv(REPRODUCTION_DIR / filename)
    pd.testing.assert_frame_equal(expected, reproduced, check_dtype=False, rtol=1e-12, atol=1e-12)

{
    "reproduction_status": "PASS",
    "fold_assignments_exact": True,
    "metric_tables_matching": len(comparison_files),
    "numeric_tolerance": "1e-12",
}

{'reproduction_status': 'PASS',
 'fold_assignments_exact': True,
 'metric_tables_matching': 7,
 'numeric_tolerance': '1e-12'}

## Reporting guardrails

- Report R² as the coefficient of determination; do not substitute Pearson *r* or *r*².
- State that splits were by site-year group and that the test groups were untouched until final evaluation.
- Describe the 18 high-CMAQ rows as quarantined pending raw-source verification; do not call them confirmed errors and do not cap them.
- Treat the no-fire-component model and the full-raw-data model as sensitivity analyses.
- State that high-regime residual error remains substantial even though the overall model improves on both the mean and linear baselines.
- The model is an error-diagnostic model, not an independently validated correction product.